# 06 · Single-Image Street-View QLoRA (`Qwen3-VL-2B-Instruct`)

Compares zero-shot vs. 1-epoch supervised fine-tuning (4-bit QLoRA) using the exact same `Qwen3-VL-2B-Instruct` base model and prompt. Loss computes answer-token cross-entropy over structured JSON (`country`, `latitude`, `longitude`); Haversine GPS error is evaluated offline.

Upload `geoguessr_sft_upload.zip` to Colab `/content`, select a GPU runtime, run all cells, download the result ZIP into `results/`, verify outputs, and terminate the runtime.


## 2026-11-20 Scaling to 6,000 Training Images (Colab T4)

Using the identical base model and QLoRA hyperparameters (original 2,000 + 4,000 additional images, 1 epoch = 750 steps, 61 min 15 sec) on the exact same 300-image validation set:

| Metric | Zero-Shot | 2,000-Image SFT | 6,000-Image SFT |
|---|---:|---:|---:|
| Country correct | 132/300 (44.0%) | 154/300 (51.3%) | 156/300 (52.0%) |
| Within 25 km | 4/300 (1.3%) | 9/300 (3.0%) | 8/300 (2.7%) |
| Within 200 km | 32/300 (10.7%) | 44/300 (14.7%) | 44/300 (14.7%) |
| Median error (valid coords) | 1,141 km | 861 km | 832 km |

Compared with 2k SFT, 6k gained 1 new ≤ 25 km hit and lost 2 prior hits. Full predictions and adapter are archived locally in `results/qlora_6000/geoguessr-sft-6000-results.zip`.


## 2026-11-20 Initial 2,000-Image QLoRA Run (Colab T4)

`Qwen3-VL-2B-Instruct`, 4-bit QLoRA (`r=8`), 2,000 training images, 1 epoch (250 steps), evaluated on the same 300 held-out images with adapter disabled (zero-shot) vs. enabled:

| Metric | Zero-Shot | 1-Epoch SFT (2k) |
|---|---:|---:|
| Country correct | 132/300 (44.00%) | 154/300 (51.33%) |
| Within 25 km | 4/300 (1.33%) | 9/300 (3.00%) |
| Within 200 km | 32/300 (10.67%) | 44/300 (14.67%) |
| Median error (valid coords) | 1,141 km | 861 km |
| Valid coordinates | 294/300 (98.00%) | 300/300 (100.00%) |

Gained 5 new ≤ 25 km hits with 0 regressions. Full predictions and adapter are archived locally in `results/qlora/geoguessr-sft-run-results.zip` (GPU runtime verified terminated).


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import shutil
import subprocess
import sys

ROOT = Path('/content/geoguessr_sft')
ROOT.mkdir(parents=True, exist_ok=True)
UPLOAD = Path('/content/geoguessr_sft_upload.zip')
PARTS = Path('/content/sft_upload_parts')
if PARTS.is_dir():
    temporary = UPLOAD.with_suffix('.part')
    with temporary.open('wb') as output:
        for part in sorted(PARTS.glob('part-*')):
            with part.open('rb') as source:
                shutil.copyfileobj(source, output)
    with temporary.open('rb') as handle:
        received = hashlib.file_digest(handle, 'sha256').hexdigest()
    if received != (PARTS / 'bundle.sha256').read_text().strip():
        raise ValueError('Upload checksum mismatch; confirm all parts have arrived.')
    temporary.replace(UPLOAD)
if not UPLOAD.is_file():
    raise FileNotFoundError(f'Upload {UPLOAD.name} or {PARTS.name} first.')
with ZipFile(UPLOAD) as archive:
    for name in ('phase1.py', 'sft_colab.py', 'requirements-colab.txt'):
        (ROOT / name).write_bytes(archive.read(name))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
                '-r', str(ROOT / 'requirements-colab.txt')], check=True)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU kernel before training.")
print(torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

In [ ]:
# Smoke check first, then run the full subset using the already-cached model.
for mode in ("smoke", "run"):
    command = [sys.executable, "-u", str(ROOT / "sft_colab.py"), str(UPLOAD),
               "--output", str(ROOT / mode),
               "--export", f"/content/geoguessr-sft-{mode}-results.zip"]
    if mode == "smoke":
        command.append("--smoke")
    with (ROOT / f"{mode}.log").open("a") as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line.rstrip(), flush=True)
            log.write(line)
            log.flush()
        if (status := proc.wait()) != 0:
            raise subprocess.CalledProcessError(status, command)

In [ ]:
import pandas as pd

metrics = pd.read_csv(ROOT / "run" / "metrics.csv")
for column in ("valid_coordinates", "country_correct", "within_25km", "within_200km"):
    metrics[f"{column}_pct"] = (100 * metrics[column] / metrics["n"]).round(2)
display(metrics)
print((ROOT / "run" / "paired_metrics.json").read_text())
print("Download /content/geoguessr-sft-run-results.zip to the SSD before terminating the runtime.")